# StrongARM Latch Comparator — Sizing & Equation Derivations

**Block:** `saradc` comparator (NanoVolt, GF180MCU `gf180mcuD`, 3.3 V devices)
**Reference:** B. Razavi, *"The StrongARM Latch: A Circuit for All Seasons,"* IEEE Solid-State Circuits Magazine, Spring 2015.
**Companion data:** `../../../Sizing/techsweep_gf180mcu_*` — the same gm/ID tables used by `techsweep_gf180mcu_plots.ipynb`.

This notebook does two things:

1. **Derives every equation** used to size the comparator — both the *system-level* budget equations (LSB, decision time, noise, power) and the *StrongARM circuit* equations (gain, regeneration time constant, offset, input-referred noise) from Razavi's column, step by step.
2. **Turns those equations into GF180 numbers** by reading `vth`, `gm/ID`, `Jd = ID/W` and `vdsat` from the technology sweep, then prints a recommended sizing table.

> **Working assumptions** (edit in the config cell): VDD = 3.3 V; NMOS-input topology (Razavi Fig. 1(b)); comparator input common-mode held at VCM ≈ 1.5 V by the sampling scheme; mismatch A_VT ≈ 5 mV·µm (placeholder). The thermal-noise factor **γ is read from the noise-complete gm/ID table** (`STH`); only A_VT still needs a PDK/Monte-Carlo extraction.

## 0. Specifications (from the SAR ADC spec sheet)

| Parameter | Value | Drives |
|---|---|---|
| Resolution N | 10 bit | LSB, noise budget |
| Sample rate f_s | 10 MS/s | decision-time budget |
| Full scale (diff) | 2.0 V_pp | LSB |
| Reference V_ref | 1.0 V | CDAC (not the latch directly) |
| SNDR | ≥ 55 dB | noise budget |
| ENOB | ≥ 9 bit | — |
| Regeneration | < 5 ns | M3–M6 sizing |
| Total ADC power | < 300 µW | comparator power share |

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import constants as sc
import gzip, shutil, tempfile
try:
    from pygmid import Lookup as lk          # gm/ID table interpolation
except ImportError as e:
    raise ImportError("pygmid missing -> in the IIC-OSIC-TOOLS container: pip install pygmid") from e

# ---- physical constants (scipy CODATA) ----
k  = sc.Boltzmann          # Boltzmann [J/K]
q  = sc.elementary_charge  # electron charge [C]
T  = 300.0                 # temperature [K] (design choice, ~27 C)
kT = k*T

# ---- ADC specs ----
N        = 10
fs       = 10e6
FS_diff  = 2.0             # full-scale differential swing [Vpp]
VREF     = 1.0
SNDR_dB  = 55.0
VDDA     = 3.3
t_reg_spec = 5e-9

# ---- design assumptions (EDIT ME) ----
VCM    = 1.65     # comparator input common mode [V]
gamma  = 0.7     # fallback only; OVERRIDDEN below by gamma = STH/(4kT*gm) read from the table
AVT_N  = 5e-3    # NMOS Vth mismatch coeff [V*um] (Pelgrom)  -> placeholder
AVT_P  = 6e-3    # PMOS Vth mismatch coeff [V*um]            -> placeholder

LSB = FS_diff/2**N
print(f"LSB (differential) = {LSB*1e3:.3f} mV")

LSB (differential) = 1.953 mV


## A. From ADC specs to comparator budgets

### A.1 LSB
$$\mathrm{LSB} = \frac{V_\mathrm{FS,diff}}{2^N} = \frac{2.0\ \mathrm{V}}{1024} = 1.95\ \mathrm{mV}.$$
Every comparator error (offset, noise, metastability) is judged against this.

### A.2 Decision-time budget
One sample period $T_s = 1/f_s = 100$ ns is shared between acquiring the input and running $N$ bit-trials. With a sampling fraction $\alpha$ (≈15–20 %) and a per-trial comparator share $\beta$ (≈0.4–0.5, the rest being DAC settling + SAR logic):
$$t_\mathrm{trial}\approx\frac{(1-\alpha)\,T_s}{N},\qquad t_\mathrm{comp}\approx\beta\,t_\mathrm{trial}.$$
$t_\mathrm{comp}$ is the wall the regeneration must beat; the "< 5 ns" spec is the explicit ceiling.

### A.3 Noise budget (the binding one)
Start from the definition of SNDR for a full-scale sine:
$$\mathrm{SNDR}=20\log_{10}\frac{V_\mathrm{sig,rms}}{V_\mathrm{n+d,rms}},\qquad V_\mathrm{sig,rms}=\frac{V_\mathrm{FS,diff}/2}{\sqrt2}.$$
Invert for the **total** allowed noise+distortion:
$$V_\mathrm{n+d,rms}=\frac{V_\mathrm{sig,rms}}{10^{\,\mathrm{SNDR}/20}}.$$
The quantizer already spends $V_{q,\mathrm{rms}}=\mathrm{LSB}/\sqrt{12}$ of that. Assuming sources add in power (uncorrelated), the budget left for **circuit** noise is
$$V_\mathrm{circ}=\sqrt{V_\mathrm{n+d,rms}^2-V_q^2}.$$
This is shared by (i) sampling $kT/C$ on the CDAC, (ii) the comparator, (iii) reference noise, (iv) DAC mismatch. The CDAC here is huge ($\sim$410 pF/side) so its $\sqrt{kT/C}$ is only a few µV — negligible. Allocate roughly half of $V_\mathrm{circ}$ to the comparator:
$$\sigma_{n,\mathrm{comp}}\lesssim\tfrac{1}{\sqrt2}\,V_\mathrm{circ}.$$

### A.4 Power budget
A StrongARM burns only dynamic energy (zero static). Each decision charges/discharges the internal nodes:
$$E_\mathrm{dec}\approx(2C_{P,Q}+C_{X,Y})\,V_\mathrm{DD}^2,\qquad P_\mathrm{comp}=E_\mathrm{dec}\,f_s\,(N+1).$$

In [2]:
# A.2 decision time
Ts = 1/fs
alpha, beta = 0.15, 0.45
t_trial = (1-alpha)*Ts/N
t_comp  = beta*t_trial
print(f"A.2  Ts={Ts*1e9:.0f} ns | t_trial~{t_trial*1e9:.2f} ns | t_comp~{t_comp*1e9:.2f} ns  (spec <{t_reg_spec*1e9:.0f} ns)")

# A.3 noise budget
Vsig  = (FS_diff/2)/np.sqrt(2)
Vnd   = Vsig/10**(SNDR_dB/20)
Vq    = LSB/np.sqrt(12)
Vcirc = np.sqrt(max(Vnd**2 - Vq**2, 0.0))
sigma_n_comp = Vcirc/np.sqrt(2)
print(f"A.3  Vsig={Vsig:.3f} Vrms | total n+d={Vnd*1e3:.3f} mV | quant={Vq*1e3:.3f} mV")
print(f"     circuit budget={Vcirc*1e3:.3f} mV -> comparator target sigma_n <= {sigma_n_comp*1e3:.3f} mV ({sigma_n_comp/LSB:.2f} LSB)")

# A.4 quick power sanity (caps refined in section D)
CPQ_g, CXY_g = 10e-15, 20e-15
E_dec = (2*CPQ_g+CXY_g)*VDDA**2
print(f"A.4  E/decision~{E_dec*1e15:.2f} fJ -> P_comp~{E_dec*fs*(N+1)*1e6:.1f} uW  (300 uW total ADC budget)")

A.2  Ts=100 ns | t_trial~8.50 ns | t_comp~3.83 ns  (spec <5 ns)
A.3  Vsig=0.707 Vrms | total n+d=1.257 mV | quant=0.564 mV
     circuit budget=1.124 mV -> comparator target sigma_n <= 0.795 mV (0.41 LSB)
A.4  E/decision~435.60 fJ -> P_comp~47.9 uW  (300 uW total ADC budget)


## B. The StrongARM equations, derived

Razavi splits one clock cycle into four phases (his Fig. 2): **(1) precharge** P,Q,X,Y → VDD; **(2) amplification** M1/M2 discharge P,Q; **(3) NMOS regeneration** M3/M4 turn on; **(4) PMOS regeneration** M5/M6 latch to the rails. Nodes: $P,Q$ are the M1/M2 drains (cap $C_{P,Q}$); $X,Y$ are the latch outputs (cap $C_{X,Y}$).

### B.1 Amplification gain — Razavi eq. (1)
In phase 2, M3–M6 are off, so M1/M2 simply **integrate** their drain currents onto $C_{P,Q}$. The *differential* signal builds as
$$|V_P-V_Q|\approx\frac{g_{m1,2}\,|V_\mathrm{in1}-V_\mathrm{in2}|}{C_{P,Q}}\,t,$$
while the *common mode* of P,Q falls at the rate set by the CM current $I_\mathrm{CM}$ drawn from each cap:
$$V_{P,Q}(t)\approx V_\mathrm{DD}-\frac{I_\mathrm{CM}}{C_{P,Q}}\,t.$$
Phase 2 ends when M3/M4 turn on, i.e. when P,Q have fallen by $V_{THN}$:
$$t_\mathrm{amp}\approx\frac{C_{P,Q}\,V_{THN}}{I_\mathrm{CM}}.$$
The gain accumulated up to that instant is
$$A_v=\frac{|V_P-V_Q|}{|V_\mathrm{in}|}=\frac{g_{m1,2}}{C_{P,Q}}\,t_\mathrm{amp}=\frac{g_{m1,2}}{C_{P,Q}}\cdot\frac{C_{P,Q}V_{THN}}{I_\mathrm{CM}}=\boxed{\;\frac{g_{m1,2}V_{THN}}{I_\mathrm{CM}}\;}\quad(\text{eq. 1}).$$
$C_{P,Q}$ cancels. Since $g_{m1,2}/I_\mathrm{CM}=g_m/I_D$, this is simply $A_v=(g_m/I_D)\,V_{THN}$ — a number you read straight off the techsweep.

### B.2 Regeneration time constant — Razavi eqs. (2)–(9)
In phase 3 (Fig. 2(d)) the cross-coupled M3/M4 drive X,Y while P,Q carry the differential current $\pm\Delta I$. KCL at the four nodes (Razavi 2–5):
$$-C_X\dot V_X=g_{m3}(V_Y-V_P),\qquad -C_Y\dot V_Y=g_{m4}(V_X-V_Q),$$
$$-C_P\dot V_P=C_X\dot V_X+\Delta I,\qquad -C_Q\dot V_Q=C_Y\dot V_Y-\Delta I.$$
Take $C_X=C_Y\equiv C_{X,Y}$, $g_{m3}=g_{m4}\equiv g_{m3,4}$ and subtract the Y-equation from the X-equation:
$$-C_{X,Y}\frac{d(V_X-V_Y)}{dt}=g_{m3,4}\big[-(V_X-V_Y)-(V_P-V_Q)\big]\quad(\text{eq. 6}).$$
Integrating the P,Q equations and combining couples the two node pairs (eq. 7):
$$C_{P,Q}(V_Q-V_P)=C_{X,Y}(V_X-V_Y)+2\Delta I\,t\;\Rightarrow\;(V_P-V_Q)=-\frac{C_{X,Y}(V_X-V_Y)+2\Delta I\,t}{C_{P,Q}}.$$
Substituting into eq. 6:
$$C_{X,Y}\frac{d(V_X-V_Y)}{dt}-g_{m3,4}\!\left(1-\frac{C_{X,Y}}{C_{P,Q}}\right)(V_X-V_Y)=-2g_{m3,4}\frac{\Delta I}{C_{P,Q}}t\quad(\text{eq. 8}).$$
The homogeneous part is a *positive-feedback* (growing) exponential $\exp(t/\tau_\mathrm{reg})$ with
$$\boxed{\;\tau_\mathrm{reg}=\frac{C_{X,Y}}{g_{m3,4}\left(1-\dfrac{C_{X,Y}}{C_{P,Q}}\right)}\;}\quad(\text{eq. 9}).$$
**Two design lessons drop out:** (i) speed wants large $g_{m3,4}$ and small $C_{X,Y}$ → keep the following RS-latch/inverter load light; (ii) the $(1-C_{X,Y}/C_{P,Q})$ term means if $C_{X,Y}\ge C_{P,Q}$ the NMOS pair barely regenerates — the PMOS pair M5/M6 then does the final latching in phase 4. Time to resolve an initial imbalance $\Delta V_0$ to a logic level $V_L$: $t_\mathrm{res}\approx\tau_\mathrm{reg}\ln(V_L/\Delta V_0)$.

### B.3 Offset — Razavi eqs. (10)–(12) + Pelgrom
Writing each drain current as CM + signal and integrating onto $C_P,C_Q$ separately:
$$V_P=V_\mathrm{DD}-\frac{g_{m1}(V_\mathrm{in1}-V_\mathrm{in2})}{2C_P}t-\frac{I_\mathrm{CM}}{C_P}t,\qquad V_Q=V_\mathrm{DD}+\frac{g_{m2}(V_\mathrm{in1}-V_\mathrm{in2})}{2C_Q}t-\frac{I_\mathrm{CM}}{C_Q}t,$$
$$\Rightarrow\;V_P-V_Q=-\frac{g_{m1,2}}{2}\frac{C_P+C_Q}{C_PC_Q}(V_\mathrm{in1}-V_\mathrm{in2})\,t+\frac{C_P-C_Q}{C_PC_Q}I_\mathrm{CM}\,t\quad(\text{eq. 12}).$$
The second term is a **deliberate** offset you program by making $C_P\ne C_Q$ — the capacitor-trim cancellation knob (Razavi Fig. 5). The **random** offset is dominated by M1/M2 mismatch (the precharge keeps M3–M6 off until gain accrues, dividing their contribution by $\sim A_v$ and $\sim10$). Pelgrom:
$$\sigma_{V_{os}}\approx\frac{A_{VT}}{\sqrt{W L}}.$$
In a SAR this offset is a *static* input shift — it does **not** degrade SNDR/DNL/INL, it only eats a little range — so the input pair is sized for matching **area**, not for a tight offset.

### B.4 Input-referred noise — Razavi eqs. (13)–(15)
In phase 2 the circuit is an **integrator**: the white thermal-noise current of M1/M2, $\overline{i_n^2}=4kT\gamma g_{m1,2}$, integrates onto $C_{P,Q}$, so the P–Q noise variance grows *linearly* with time (eq. 13):
$$\overline{V_{PQ}^2}=\frac{8kT\gamma g_{m1,2}}{C_{P,Q}^2}\,t.$$
Evaluate at the end of amplification $t=t_\mathrm{amp}=C_{P,Q}V_{THN}/I_\mathrm{CM}$ (eq. 14):
$$\sigma_{1,2}^2=\frac{8kT\gamma g_{m1,2}}{C_{P,Q}^2}\cdot\frac{C_{P,Q}V_{THN}}{I_\mathrm{CM}}=\frac{8kT\gamma g_{m1,2}V_{THN}}{C_{P,Q}I_\mathrm{CM}}.$$
Refer to the input by dividing by $A_v^2=(g_{m1,2}V_{THN}/I_\mathrm{CM})^2$ and use $g_{m1,2}\approx 2I_\mathrm{CM}/(V_{GS}-V_{THN})$:
$$\frac{\sigma_{1,2}^2}{A_v^2}=\frac{8kT\gamma I_\mathrm{CM}}{C_{P,Q}g_{m1,2}V_{THN}}=\frac{V_{GS}-V_{THN}}{V_{THN}}\cdot\frac{4kT\gamma}{C_{P,Q}}.$$
Adding the $kT/C$ deposited by the precharge switches S1/S2 (referred to input) gives eq. (15):
$$\boxed{\;\overline{V_{n,\mathrm{in}}^2}=\frac{V_{GS}-V_{THN}}{V_{THN}}\left[\frac{4kT\gamma}{C_{P,Q}}+\frac{V_{GS}-V_{THN}}{V_{THN}}\frac{kT}{2C_{P,Q}}\right]\;}\quad(\text{eq. 15}).$$
**Design reading:** lower input overdrive $(V_{GS}-V_{THN})$ and larger $C_{P,Q}$ both cut noise; the first (thermal) term usually dominates. Inverting eq. (15) gives the $C_{P,Q}$ you need to hit $\sigma_{n,\mathrm{comp}}$.

In [3]:
# --- the four StrongARM equations as functions ---
def av_gain(gm_id, Vthn):
    "Razavi eq.1:  Av = gm*Vthn/Icm = (gm/Id)*Vthn"
    return gm_id*Vthn

def t_amp(Cpq, Vthn, Icm):
    return Cpq*Vthn/Icm

def tau_reg(Cxy, gm34, Cpq):
    "Razavi eq.9"
    return Cxy/(gm34*(1-Cxy/Cpq))

def vn_in2(Vov, Vthn, Cpq, gamma=gamma):
    "Razavi eq.15 -> input-referred noise variance [V^2]"
    r = Vov/Vthn
    return r*(4*kT*gamma/Cpq + r*kT/(2*Cpq))

def cpq_for_noise(sigma_n, Vov, Vthn, gamma=gamma):
    "invert eq.15 for the Cpq needed to hit a noise target"
    r = Vov/Vthn
    return (r*4*kT*gamma + r**2*kT/2)/sigma_n**2

def offset_pelgrom(AVT, W_um, L_um):
    return AVT/np.sqrt(W_um*L_um)

print("equation helpers defined")

equation helpers defined


## C. GF180 device data via pygmid (gm/ID lookup tables)

We use the **pygmid** `Lookup` class — same flow as `sky130_gff_ip__bgr_1.2v/sizing/sizing_op.ipynb` — on the GF180 `.mat` tables in `../../../Sizing/`. For each device pick a $g_m/I_D$ regime and $L$, then look up the operating point **directly**:

- `lookupVGS(GM_ID=., L=., VDS=., VSB=.)` → $V_{GS}$
- `lookup('ID_W' / 'VT', ...)` → current density [A/µm] and threshold; then $W = I_D/(I_D/W)$

For diode-like devices (the cross-coupled pairs) the self-consistent bias has $V_{DS}=V_{GS}$, found with the two-line idiom from the reference:
```
vgs = n.lookupVGS(GM_ID=gmid, L=L, VDS=0.9, VSB=0)   # first estimate
vds = n.lookupVGS(GM_ID=gmid, L=L, VDS=vgs, VSB=0)   # re-lookup at VDS = VGS
```
$L$ is in **µm**. These tables are **noise-complete** (`STH`, `SFL` populated by the re-run techsweep), so we read the thermal-noise factor $\gamma = S_\mathrm{th}/(4kT\,g_m)$ straight from the table rather than assuming it. High $g_m/I_D$ (low overdrive) → low noise for the input pair; low $g_m/I_D$ (strong inversion) → high regeneration $g_m$ for the cross-coupled pairs.

In [4]:
# --- locate Sizing/ and load the pygmid tables (decompress .mat.gz on the fly) ---
CANDS = ['../../../Sizing',
         '/foss/designs/nanovolt-chipathon2026/designs/Sizing',
         str(Path.home()/'eda/designs/nanovolt-chipathon2026/designs/Sizing')]
SIZING = next((Path(c) for c in CANDS
               if (Path(c)/'gf180mcu_nmos_3v3.mat').exists()
               or (Path(c)/'gf180mcu_nmos_3v3.mat.gz').exists()), None)
assert SIZING is not None, "could not locate Sizing/ .mat tables"

CACHE = Path(tempfile.gettempdir())/'gmid_gf180'; CACHE.mkdir(exist_ok=True)
def ensure_mat(stem):
    src = SIZING/f'{stem}.mat'
    if src.exists():
        return str(src)
    dst = CACHE/f'{stem}.mat'
    if not dst.exists():
        with gzip.open(SIZING/f'{stem}.mat.gz','rb') as fi, open(dst,'wb') as fo:
            shutil.copyfileobj(fi, fo)
    return str(dst)

n = lk(ensure_mat('gf180mcu_nmos_3v3'))
p = lk(ensure_mat('gf180mcu_pmos_3v3'))
print("loaded GF180 nmos/pmos gm/ID tables from", SIZING)

def scal(x):                       # pygmid may return list / ndarray / scalar -> force a float
    return float(np.asarray(x).ravel()[0])

# --- pick gm/ID regime + L per device, then look up the operating point ---
# GM_ID -> VGS via lookupVGS, then fully-specified (L,VGS,VDS,VSB) mode-1 lookups return scalars.
# input pair M1/M2 : high gm/ID -> low noise ; biased near VDS = VDDA/2
gmid_in, L_in = 15, 0.5
vgs_in = scal(n.lookupVGS(GM_ID=gmid_in, L=L_in, VDS=VDDA/2, VSB=0.0))
vth_in = scal(n.lookup('VT',   L=L_in, VGS=vgs_in, VDS=VDDA/2, VSB=0.0))
idw_in = scal(n.lookup('ID_W', L=L_in, VGS=vgs_in, VDS=VDDA/2, VSB=0.0))   # A/um
vov_in = vgs_in - vth_in

# cross-coupled NMOS M3/M4 : strong inversion -> fast regen ; diode-like so VDS = VGS
gmid_xn, L_xn = 10, 0.28
vgs_xn = scal(n.lookupVGS(GM_ID=gmid_xn, L=L_xn, VDS=0.9,    VSB=0.0))     # first estimate
vds_xn = scal(n.lookupVGS(GM_ID=gmid_xn, L=L_xn, VDS=vgs_xn, VSB=0.0))     # re-lookup at VDS = VGS
idw_xn = scal(n.lookup('ID_W', L=L_xn, VGS=vds_xn, VDS=vds_xn, VSB=0.0))   # diode point VGS=VDS

# cross-coupled PMOS M5/M6
gmid_xp, L_xp = 10, 0.28
vgs_xp = scal(p.lookupVGS(GM_ID=gmid_xp, L=L_xp, VDS=0.9,    VSB=0.0))
vds_xp = scal(p.lookupVGS(GM_ID=gmid_xp, L=L_xp, VDS=vgs_xp, VSB=0.0))
idw_xp = scal(p.lookup('ID_W', L=L_xp, VGS=vds_xp, VDS=vds_xp, VSB=0.0))

print(f"M1/M2 in : gm/Id={gmid_in} L={L_in}um  Vgs={vgs_in:.3f} Vth={vth_in:.3f} Vov={vov_in:+.3f}  Id/W={idw_in*1e6:.2f} uA/um")
print(f"M3/M4 xn : gm/Id={gmid_xn} L={L_xn}um  Vgs={vgs_xn:.3f} Vds={vds_xn:.3f}  Id/W={idw_xn*1e6:.2f} uA/um")
print(f"M5/M6 xp : gm/Id={gmid_xp} L={L_xp}um  Vgs={vgs_xp:.3f} Vds={vds_xp:.3f}  Id/W={idw_xp*1e6:.2f} uA/um")

# --- thermal-noise factor gamma straight from the noise-complete table (STH) ---
gm_in  = scal(n.lookup('GM',  L=L_in, VGS=vgs_in, VDS=VDDA/2, VSB=0.0))
sth_in = scal(n.lookup('STH', L=L_in, VGS=vgs_in, VDS=VDDA/2, VSB=0.0))   # S_id thermal PSD [A^2/Hz]
gamma_tab = sth_in/(4*kT*gm_in)
if 0.3 < gamma_tab < 2.0:
    gamma = gamma_tab
    print(f"gamma (from STH) = {gamma:.3f}   [thermal-noise factor at the input-pair OP]")
else:
    print(f"[warn] gamma from table = {gamma_tab:.3g} implausible (STH zero/missing?) -> keep fallback gamma={gamma}")

loaded GF180 nmos/pmos gm/ID tables from ../../../Sizing
M1/M2 in : gm/Id=15 L=0.5um  Vgs=0.745 Vth=0.710 Vov=+0.035  Id/W=1.35 uA/um
M3/M4 xn : gm/Id=10 L=0.28um  Vgs=0.757 Vds=0.761  Id/W=6.75 uA/um
M5/M6 xp : gm/Id=10 L=0.28um  Vgs=0.882 Vds=0.882  Id/W=2.50 uA/um
gamma (from STH) = 0.720   [thermal-noise factor at the input-pair OP]


## D. Equations → sizing numbers
Now we feed the GF180 operating points into the boxed equations.

In [5]:
Vthn = vth_in

# D.1  noise -> required Cpq  (invert eq.15)
Cpq_req = cpq_for_noise(sigma_n_comp, vov_in, Vthn, gamma)
CXY = 20e-15                                    # X,Y load estimate (refine from extraction)
CPQ = max(Cpq_req, 1.5*CXY, 10e-15)             # keep Cpq > Cxy so eq.9 regenerates
print(f"D.1 NOISE : Vov_in={vov_in:+.3f} V, Vthn={Vthn:.3f} V, gamma={gamma:.3f} -> Cpq_req={Cpq_req*1e15:.1f} fF; "
      f"choose Cpq={CPQ*1e15:.0f} fF (sigma_n={np.sqrt(vn_in2(vov_in,Vthn,CPQ,gamma))*1e3:.2f} mV)")

# D.3  input pair: area = max(offset-driven, current-driven)
Icm    = CPQ*Vthn/t_comp                          # per-side CM current within the comparator window
gm12   = gmid_in*Icm
W_in_i = Icm/idw_in                               # width to carry Icm at this gm/Id
WL_req = (AVT_N/5e-3)**2                           # area for sigma_os = 5 mV
W_in_o = WL_req/L_in
W_in   = max(W_in_i, W_in_o)
print(f"D.3 INPUT : Icm={Icm*1e6:.1f}uA gm12={gm12*1e6:.0f}uS -> W(current)={W_in_i:.1f}um | "
      f"W(offset)={W_in_o:.1f}um -> W_in={W_in:.1f}um @L={L_in}um (sigma_os={offset_pelgrom(AVT_N,W_in,L_in)*1e3:.2f}mV)")

# D.2  regen -> gm34 -> W34  (eq.9)
n_tc       = np.log((VDDA/2)/1e-3)
tau_target = t_comp/(n_tc + 2)
gm34       = CXY/(tau_target*(1-CXY/CPQ))
id34       = gm34/gmid_xn
W34        = id34/idw_xn
W56        = W34*idw_xn/idw_xp                     # PMOS scaled for matched strength
print(f"D.2 REGEN : tau_target={tau_target*1e12:.0f}ps -> gm34={gm34*1e6:.0f}uS -> id34={id34*1e6:.1f}uA "
      f"-> W34={W34:.1f}um, W56={W56:.1f}um (tau_reg={tau_reg(CXY,gm34,CPQ)*1e12:.0f}ps)")

# D.4  gain & power
E_dec  = (2*CPQ+CXY)*VDDA**2
P_comp = E_dec*fs*(N+1)
print(f"D.4 GAIN/PWR: Av=(gm/Id)*Vthn={av_gain(gmid_in,Vthn):.1f} V/V | "
      f"E/dec={E_dec*1e15:.2f}fJ -> P_comp~{P_comp*1e6:.1f}uW (of 300uW total)")

D.1 NOISE : Vov_in=+0.035 V, Vthn=0.710 V, gamma=0.720 -> Cpq_req=0.9 fF; choose Cpq=30 fF (sigma_n=0.14 mV)
D.3 INPUT : Icm=5.6uA gm12=84uS -> W(current)=4.1um | W(offset)=2.0um -> W_in=4.1um @L=0.5um (sigma_os=3.49mV)
D.2 REGEN : tau_target=407ps -> gm34=148uS -> id34=14.8uA -> W34=2.2um, W56=5.9um (tau_reg=407ps)
D.4 GAIN/PWR: Av=(gm/Id)*Vthn=10.6 V/V | E/dec=871.20fJ -> P_comp~95.8uW (of 300uW total)


In [6]:
snap = lambda w: max(round(w*2)/2, 0.5)      # round to 0.5 um grid, >= 0.5 um
sizing = pd.DataFrame([
    dict(Device='M1,M2 input',     W_um=snap(W_in), L_um=L_in, nf=4, gm_id=gmid_in, set_by='offset area + input gm'),
    dict(Device='M7 tail',         W_um=6.0,        L_um=0.28, nf=4, gm_id='-',     set_by='switch: low Ron, limit kickback'),
    dict(Device='M3,M4 x-NMOS',    W_um=snap(W34),  L_um=L_xn, nf=2, gm_id=gmid_xn, set_by='regeneration gm (eq.9)'),
    dict(Device='M5,M6 x-PMOS',    W_um=snap(W56),  L_um=L_xp, nf=2, gm_id=gmid_xp, set_by='matched pull-up; restore VDD'),
    dict(Device='M8-M11 precharge',W_um=2.0,        L_um=0.28, nf=1, gm_id='-',     set_by='precharge P,Q,X,Y in reset'),
])
print(sizing.to_string(index=False))
print(f"\nSUMMARY: LSB={LSB*1e3:.2f}mV | sigma_n<= {sigma_n_comp*1e3:.2f}mV | "
      f"Cpq={CPQ*1e15:.0f}fF | Av~{av_gain(gmid_in,Vthn):.1f} | P_comp~{P_comp*1e6:.1f}uW")

          Device  W_um  L_um  nf gm_id                          set_by
     M1,M2 input   4.0  0.50   4    15          offset area + input gm
         M7 tail   6.0  0.28   4     - switch: low Ron, limit kickback
    M3,M4 x-NMOS   2.0  0.28   2    10          regeneration gm (eq.9)
    M5,M6 x-PMOS   6.0  0.28   2    10    matched pull-up; restore VDD
M8-M11 precharge   2.0  0.28   1     -      precharge P,Q,X,Y in reset

SUMMARY: LSB=1.95mV | sigma_n<= 0.79mV | Cpq=30fF | Av~10.6 | P_comp~95.8uW


## E. Caveats & next steps

- **γ is now read from the table** (`STH`, from the noise-complete re-characterization — see `Sizing/` and `gmid_murmann/README.md`). Only **A_VT remains a placeholder** — get it from a PDK mismatch model / Monte-Carlo. Flicker (`SFL`/`SFL_GM`) is also in the table if you later need input-referred 1/f.
- pygmid **interpolates** L/VDS/VSB, but stay near grid: L ∈ {0.28, 0.5, 1, 2, 5, 10} µm, VSB ∈ {0, 0.4, 0.8, 1.2} V. The input-pair source sits at the tail node (VSB > 0 in operation) — re-check Vth/Vov with the real VSB once known.
- The input-pair bias is **dynamic**; the gm/ID operating point is only a proxy to convert $g_m\leftrightarrow W$. Confirm everything in transient.
- Build these testbenches in `saradc/tb/`:
  1. **Transient regeneration** — clock it, sweep $\Delta V_\mathrm{in}$ from ±LSB/2 down to ±0.1 mV; confirm rail-to-rail resolve < 5 ns and find the metastable point.
  2. **Offset** — short inputs, Monte-Carlo mismatch (≥200 pts); read $\sigma_{os}$.
  3. **Input-referred noise** — Razavi's clocked method (Fig. 6): zero offset, apply small $V_S$, clock many times with transient noise on, count 1s/0s vs $V_S$, back out $\sigma_{n,\mathrm{in}}$.
  4. **Kickback** — drive M1/M2 gates from a realistic CDAC source impedance; measure input disturbance on CK edges.
  5. **Corners** — SS/FF/SF/FS, 3.0–3.6 V, −40/27/125 °C.
- Confirm the comparator input CM during the LSB trials stays $\gtrsim V_{THN}+V_{dsat,\mathrm{tail}}$; if it droops, move to VCM-based DAC switching or a **double-tail** dynamic comparator.